# Nauka o Danych I — Laboratorium 2
## Praktyczne zastosowanie podstawowych funkcji statystycznych w analizie danych

**Wariant 14:** United States Health-Care Spending Attributable to Modifiable Risk Factors 2016  
**Środowisko:** Python, Jupyter Notebook, `numpy`, `pandas`, `matplotlib`  
**Dane:** `IHME_USA_RISK_SPENDING_2016_Y2020M09D29.csv`

Cel: zastosować sześć miar opisanych w instrukcji laboratorium: **średnią arytmetyczną, medianę, odchylenie standardowe, wariancję, korelację Pearsona i kowariancję**. Analizujemy liczby z kolumn `mean`, `lower` i `upper`.

**Uwaga o interpretacji:** kolumna `mean` zawiera oszacowaną wartość wydatków dla poszczególnych rekordów badania; obliczana poniżej średnia to *średnia arytmetyczna tych oszacowań w wierszach tabeli*, a nie całkowity koszt systemu ochrony zdrowia. Poszczególne rekordy mogą reprezentować różne przekroje danych.

### Instalacja (jednorazowo, w PowerShell)

```powershell
py -m pip install notebook numpy pandas matplotlib
py -m notebook
```

Plik CSV musi być w tym samym katalogu co notebook. Uruchamiaj komórki kolejno przez **Shift+Enter**, a po zakończeniu zapisz wyniki przez **Ctrl+S**.

## 1. Import bibliotek i przygotowanie danych

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

plik = Path("IHME_USA_RISK_SPENDING_2016_Y2020M09D29.csv")
if not plik.exists():
    raise FileNotFoundError(f"Umieść plik {plik.name} obok notebooka: {Path.cwd()}")

df = pd.read_csv(plik)
print(f"Wczytano {df.shape[0]} wierszy i {df.shape[1]} kolumn")
display(df.head(5))

In [ ]:
print("Typy danych i brakujące wartości:")
display(pd.DataFrame({"typ": df.dtypes.astype(str), "braki": df.isna().sum()}))

# Zmienne liczbowe służące do obliczeń statystycznych
kolumny = ["mean", "lower", "upper"]
dane = df[kolumny].apply(pd.to_numeric, errors="coerce").dropna()
x = dane["mean"].to_numpy()
y = dane["upper"].to_numpy()
print("Liczba kompletnych obserwacji:", len(dane))
print("Jednostka:", df["metric"].iloc[0])

## 2.1. Średnia arytmetyczna (Mean)

\(\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i\)
rac{1}{n}\sum_{i=1}^n x_i\)

Średnia opisuje przeciętną wartość w analizowanej kolumnie. Obliczamy ją na rekordach kolumny `mean`, a wynik porównujemy z implementacją w `pandas`.

In [ ]:
srednia = np.mean(x)
srednia_recznie = np.sum(x) / len(x)
print(f"Średnia z kolumny 'mean': {srednia:,.4f}")
print(f"Ręcznie (suma / liczba rekordów): {srednia_recznie:,.4f}")
print(f"Porównanie z pandas: {df['mean'].mean():,.4f}")
assert np.isclose(srednia, srednia_recznie)

## 2.2. Mediana (Median)

Mediana to środkowa wartość uporządkowanego zbioru (dla parzystej liczby obserwacji: średnia dwóch środkowych). Jest odporniejsza od średniej na bardzo wysokie wartości. Porównamy ją ze średnią.

In [ ]:
mediana = np.median(x)
print(f"Mediana: {mediana:,.4f}")
print(f"Średnia: {srednia:,.4f}")
print(f"Różnica średnia − mediana: {srednia - mediana:,.4f}")
print("Liczba obserwacji:", len(x))
print("Mediana pandas:", f"{df['mean'].median():,.4f}")

## 2.3. Odchylenie standardowe (Standard Deviation)

\(\sigma=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2}\)
rac{1}{n}\sum_{i=1}^{n}(x_i-ar{x})^2}\)

W instrukcji zastosowano wzór **populacyjny** (dzielenie przez `n`). Dlatego w NumPy ustawiamy `ddof=0`; `pandas.Series.std()` domyślnie liczy wersję próbkową (`ddof=1`).

In [ ]:
odchylenie = np.std(x, ddof=0)
odchylenie_recznie = np.sqrt(np.mean((x - srednia) ** 2))
print(f"Odchylenie standardowe populacji: {odchylenie:,.4f}")
print(f"Obliczenie ze wzoru: {odchylenie_recznie:,.4f}")
print(f"Odchylenie dla próby (ddof=1): {np.std(x, ddof=1):,.4f}")
assert np.isclose(odchylenie, odchylenie_recznie)

## 2.4. Wariancja (Variance)

\(\mathrm{Var}(X)=\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2\)
rac{1}{n}\sum_{i=1}^{n}(x_i-ar{x})^2\)

Wariancja jest kwadratem odchylenia standardowego i opisuje rozproszenie danych. Jej jednostka jest podniesiona do kwadratu.

In [ ]:
wariancja = np.var(x, ddof=0)
print(f"Wariancja populacji: {wariancja:,.4f}")
print(f"Odchylenie standardowe do kwadratu: {odchylenie ** 2:,.4f}")
assert np.isclose(wariancja, odchylenie ** 2)

## 2.5. Korelacja Pearsona (Correlation)

\(r_{X,Y}=\frac{\mathrm{Cov}(X,Y)}{\sigma_X\sigma_Y}\)
rac{\mathrm{Cov}(X,Y)}{\sigma_X\sigma_Y}\)

Sprawdzamy zależność liniową między `mean` (oszacowanie centralne) a `upper` (górna granica przedziału oszacowania). Wynik jest od −1 do 1: dodatni oznacza wspólny wzrost, a ujemny przeciwny kierunek zmian. **Korelacja nie oznacza przyczynowości.**

In [ ]:
korelacja = np.corrcoef(x, y)[0, 1]
print(f"Korelacja Pearsona (mean, upper): {korelacja:.6f}")
print(f"Weryfikacja pandas: {dane['mean'].corr(dane['upper']):.6f}")
assert np.isclose(korelacja, dane['mean'].corr(dane['upper']))

## 2.6. Kowariancja (Covariance)

\(\mathrm{Cov}(X,Y)=\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})\)
rac{1}{n}\sum_{i=1}^{n}(x_i-ar{x})(y_i-ar{y})\)

Obliczamy kowariancję populacyjną (`bias=True` daje dzielenie przez `n`, zgodnie z instrukcją). Znaczenie znaku kowariancji jest podobne do znaku korelacji, ale jej wielkość zależy od skali danych.

In [ ]:
macierz_kowariancji = np.cov(x, y, bias=True)
kowariancja = macierz_kowariancji[0, 1]
kowariancja_recznie = np.mean((x - np.mean(x)) * (y - np.mean(y)))
korelacja_z_kowariancji = kowariancja / (np.std(x, ddof=0) * np.std(y, ddof=0))
print("Macierz kowariancji (populacyjnej):")
display(pd.DataFrame(macierz_kowariancji, index=["mean", "upper"], columns=["mean", "upper"]))
print(f"Kowariancja: {kowariancja:,.4f}")
print(f"Kowariancja ze wzoru: {kowariancja_recznie:,.4f}")
print(f"Korelacja wyznaczona z kowariancji: {korelacja_z_kowariancji:.6f}")
assert np.isclose(kowariancja, kowariancja_recznie)
assert np.isclose(korelacja, korelacja_z_kowariancji)

## 3. Podsumowanie wyników wszystkich sześciu miar

In [ ]:
podsumowanie = pd.DataFrame({
    "Miara": ["Średnia (mean)", "Mediana (mean)", "Odchylenie standardowe (mean)",
              "Wariancja (mean)", "Korelacja (mean, upper)", "Kowariancja (mean, upper)"],
    "Wynik": [srednia, mediana, odchylenie, wariancja, korelacja, kowariancja]
})
display(podsumowanie)
print("Wniosek:")
if srednia > mediana:
    print("Średnia jest wyższa od mediany, co sugeruje wpływ dużych wartości na rozkład.")
else:
    print("Średnia nie przekracza mediany; należy sprawdzić kształt rozkładu.")
print(f"Korelacja mean z upper wynosi {korelacja:.4f}.")

## 4. Przedstawienie wyników na wykresach

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(x, bins=35, edgecolor="white")
ax.axvline(srednia, linestyle="--", linewidth=2, label=f"Średnia: {srednia:,.0f}")
ax.axvline(mediana, linestyle=":", linewidth=2, label=f"Mediana: {mediana:,.0f}")
ax.set_title("Rozkład wartości 'mean' — wariant 14")
ax.set_xlabel("Oszacowane wydatki (2016 USD)")
ax.set_ylabel("Liczba rekordów")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, s=12, alpha=0.45)
ax.set_title(f"Zależność mean i upper (r = {korelacja:.4f})")
ax.set_xlabel("mean — oszacowanie centralne (2016 USD)")
ax.set_ylabel("upper — górna granica (2016 USD)")
plt.tight_layout()
plt.show()

## 5. Wnioski do sprawozdania

1. Wczytano oryginalne dane **wariantu 14** i przeanalizowano wartości liczbowe związane z oszacowanymi wydatkami zdrowotnymi w USA w 2016 r.
2. Średnia i mediana służą do opisu tendencji centralnej. Porównanie tych wartości pokazuje, czy rozkład może być asymetryczny.
3. Odchylenie standardowe i wariancja opisują rozproszenie rekordów. Zastosowano wzory populacyjne (`ddof=0`) z instrukcji.
4. Korelacja Pearsona i kowariancja między `mean` a `upper` pozwalają ocenić współzmienność oszacowania centralnego i górnej granicy.
5. Wykres histogramu oraz wykres punktowy ułatwiają interpretację rozkładu i zależności.

**Ograniczenie:** dane obejmują różne czynniki ryzyka, grupy wieku i płci; prosta średnia po wierszach nie jest sumą wydatków USA ani średnią ważoną populacją. Analizowane `mean` i `upper` są konstrukcyjnie powiązane, dlatego wysoka korelacja jest oczekiwana i nie stanowi dowodu związku przyczynowego.